In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_parquet("/Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/splits/train.parquet")

rank_cols = [col for col in train_df.columns if '_rank' in col]
train_df = train_df.drop(columns=rank_cols)

test_df = pd.read_parquet("/Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/splits/test.parquet")

rank_cols = [col for col in test_df.columns if '_rank' in col]
test_df = test_df.drop(columns=rank_cols)

val_df = pd.read_parquet("/Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/splits/val.parquet")

rank_cols = [col for col in val_df.columns if '_rank' in col]
val_df = val_df.drop(columns=rank_cols)

In [4]:
def winsorize_cs(group, cols, clip=5):
    
    for col in cols:
        mu = group[col].mean()
        sigma = group[col].std()
        if sigma > 0:
            group[col] = np.clip(
                group[col],
                mu - clip*sigma,
                mu + clip*sigma
            )
    return group

In [5]:
def zscore_cs(group, cols):
    for col in cols:
        mu = group[col].mean()
        sigma = group[col].std()
        if sigma > 0:
            group[col] = (group[col] - mu) / sigma
    return group


In [6]:
feature_cols = [c for c in train_df.columns 
                if c not in ["date", "ticker", "fwd_return_5d"]]

Cross-Sectionally applying both functions to all 3 dfs

In [7]:
train_df = train_df.groupby("date", group_keys=False)\
    .apply(lambda x: winsorize_cs(x, feature_cols))

train_df = train_df.groupby("date", group_keys=False)\
    .apply(lambda x: zscore_cs(x, feature_cols))
    
val_df = val_df.groupby("date", group_keys=False)\
    .apply(lambda x: winsorize_cs(x, feature_cols))

val_df = val_df.groupby("date", group_keys=False)\
    .apply(lambda x: zscore_cs(x, feature_cols))


test_df = test_df.groupby("date", group_keys=False)\
    .apply(lambda x: winsorize_cs(x, feature_cols))

test_df = test_df.groupby("date", group_keys=False)\
    .apply(lambda x: zscore_cs(x, feature_cols))

/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_46296/1552191346.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: winsorize_cs(x, feature_cols))
/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_46296/1552191346.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: zscore_cs(x, feature_cols))
/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_46

------

In [8]:
train_df.shape

(109284, 37)

In [14]:
check = train_df.groupby("date")[feature_cols].mean().mean()

check_std = train_df.groupby("date")[feature_cols].std().mean()

In [15]:
check

dollar_volume         2.651915e-18
price_ma_ratio_5     -1.090228e-18
price_ma_ratio_10    -1.966021e-18
price_ma_ratio_21     1.429179e-19
price_ma_ratio_63    -9.394514e-19
price_ma_ratio_252    1.119751e-18
pos_days_21          -1.625850e-16
z_price_21            2.025007e-19
z_price_63            4.114686e-18
z_return_1d_5         8.320588e-19
z_return_1d_21       -3.370640e-18
z_return_1d_35       -1.814107e-19
z_return_5d_5         1.095186e-18
z_return_5d_21        1.684618e-19
z_return_5d_35       -2.952779e-18
rev_1d               -1.298163e-18
rev_5d               -7.481010e-19
vol_5                 2.250047e-18
vol_10                3.850402e-18
vol_21                4.172371e-18
vol_63               -2.728538e-18
vol_ratio_21_63      -1.235150e-17
vol_ratio_5_21       -5.423393e-18
hl_range             -2.945666e-18
downside_vol_1d      -4.516661e-18
downside_vol_5d       3.079607e-18
vol_avg_5             5.714018e-19
vol_avg_21           -4.040472e-20
vol_avg_63          

In [16]:
check_std

dollar_volume         1.0
price_ma_ratio_5      1.0
price_ma_ratio_10     1.0
price_ma_ratio_21     1.0
price_ma_ratio_63     1.0
price_ma_ratio_252    1.0
pos_days_21           1.0
z_price_21            1.0
z_price_63            1.0
z_return_1d_5         1.0
z_return_1d_21        1.0
z_return_1d_35        1.0
z_return_5d_5         1.0
z_return_5d_21        1.0
z_return_5d_35        1.0
rev_1d                1.0
rev_5d                1.0
vol_5                 1.0
vol_10                1.0
vol_21                1.0
vol_63                1.0
vol_ratio_21_63       1.0
vol_ratio_5_21        1.0
hl_range              1.0
downside_vol_1d       1.0
downside_vol_5d       1.0
vol_avg_5             1.0
vol_avg_21            1.0
vol_avg_63            1.0
dollar_vol_log        1.0
volume_trend          1.0
mom_x_vol_5           1.0
mom_x_vol_21          1.0
mom_x_vol_63          1.0
dtype: float64

In [9]:
val_df.shape

(15000, 37)

In [17]:
check = val_df.groupby("date")[feature_cols].mean().mean()

check_std = val_df.groupby("date")[feature_cols].std().mean()

In [18]:
check

dollar_volume         6.174315e-19
price_ma_ratio_5     -5.970803e-18
price_ma_ratio_10     1.836320e-18
price_ma_ratio_21     1.614840e-18
price_ma_ratio_63    -5.428094e-19
price_ma_ratio_252   -1.038532e-18
pos_days_21           1.763775e-18
z_price_21            2.839258e-18
z_price_63           -8.241671e-18
z_return_1d_5         8.962478e-18
z_return_1d_21       -2.356159e-18
z_return_1d_35        4.261955e-18
z_return_5d_5        -2.641695e-18
z_return_5d_21        1.021391e-18
z_return_5d_35        4.190427e-18
rev_1d                1.413684e-18
rev_5d               -4.259382e-18
vol_5                -9.243185e-20
vol_10                1.052449e-18
vol_21                3.920070e-18
vol_63               -6.768081e-18
vol_ratio_21_63       1.547573e-17
vol_ratio_5_21        5.282406e-18
hl_range              9.197200e-18
downside_vol_1d       2.058542e-17
downside_vol_5d      -1.497500e-18
vol_avg_5             1.936530e-19
vol_avg_21            1.715367e-18
vol_avg_63          

In [10]:
test_df.shape

(14240, 37)

In [19]:
check_std

dollar_volume         1.0
price_ma_ratio_5      1.0
price_ma_ratio_10     1.0
price_ma_ratio_21     1.0
price_ma_ratio_63     1.0
price_ma_ratio_252    1.0
pos_days_21           1.0
z_price_21            1.0
z_price_63            1.0
z_return_1d_5         1.0
z_return_1d_21        1.0
z_return_1d_35        1.0
z_return_5d_5         1.0
z_return_5d_21        1.0
z_return_5d_35        1.0
rev_1d                1.0
rev_5d                1.0
vol_5                 1.0
vol_10                1.0
vol_21                1.0
vol_63                1.0
vol_ratio_21_63       1.0
vol_ratio_5_21        1.0
hl_range              1.0
downside_vol_1d       1.0
downside_vol_5d       1.0
vol_avg_5             1.0
vol_avg_21            1.0
vol_avg_63            1.0
dollar_vol_log        1.0
volume_trend          1.0
mom_x_vol_5           1.0
mom_x_vol_21          1.0
mom_x_vol_63          1.0
dtype: float64

In [20]:
from config import PROCESSED_DATA_PATH
from pathlib import Path

def save_features(df):
    
    path = Path(PROCESSED_DATA_PATH) / "train_normalized_df.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved features to {path}")

save_features(train_df)

Saved features to /Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/train_normalized_df.parquet


In [21]:
from config import PROCESSED_DATA_PATH
from pathlib import Path

def save_features(df):
    
    path = Path(PROCESSED_DATA_PATH) / "val_normalized_df.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved features to {path}")

save_features(val_df)

Saved features to /Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/val_normalized_df.parquet


In [22]:
from config import PROCESSED_DATA_PATH
from pathlib import Path

def save_features(df):
    
    path = Path(PROCESSED_DATA_PATH) / "test_normalized_df.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved features to {path}")

save_features(test_df)

Saved features to /Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/test_normalized_df.parquet


-------